In [ ]:
import math
import numpy as np
import pandas as pd

df = pd.read_csv("M_events_jobs.csv",sep=",")
print("jobs expired:",len(df[df["user_expired"].notna()]))
df = df[df["user_completed"].isna()]

df.drop(columns=["user_completed","user_expired","comp","due",'Unnamed: 0',"user_server_assigned","user_lock_for_resources","user_unlock_for_resources"], inplace=True)
# display(df)
times_dict = {}
for indx,row in df.iterrows():
    times = []
    for col in df.columns:
        if  pd.isna(row[col]):
            continue
        elif isinstance(row[col],(int,float)):
            times.append((row[col],col))
        else:
            times.extend([(float(v),col) for v in row[col].split(",")])
    times_dict[indx] = sorted(times,key=lambda x:x[0],reverse=True)
print(times_dict)

data = {"job":[],"time":[],"event":[]}
for k in times_dict:
    data["job"].append(k)
    # pasar de lista de tuplas a lista de dos listas
    lasttime = times_dict[k][0][0]
    repitedtime = [t for t in times_dict[k] if t[0] == lasttime]
    d = len(repitedtime)
    print(d)
    if d == 1:
        data["time"].append(times_dict[k][0][0])
        data["event"].append(times_dict[k][0][1])
    else:
        data["time"].append(";".join([str(x[0]) for x in times_dict[k][:d]]))
    #data["event"].append(";".join([str(x[1]) for x in times_dict[k][:3]]))

pd.DataFrame(data).to_csv("Summary_events_jobs_times.csv",index=False) 

jobs expired: 794
{0: [(18.0, 'user_rejected_all_servers_out'), (15.0, 'user_rejected_all_servers_out'), (14.0, 'user_rejected_all_servers_out'), (12.0, 'user_rejected_all_servers_out'), (11.0, 'user_rejected_all_servers_out'), (8.0, 'user_rejected_all_servers_out'), (6.0, 'user_rejected_all_servers_out'), (4.0, 'user_rejected_all_servers_out'), (3.0, 'user_rejected_all_servers_out'), (1.0, 'user_arrival'), (1.0, 'user_rejected_all_servers_out')], 1: [(96.0, 'user_rejected_all_servers_out'), (95.0, 'user_rejected_all_servers_out'), (93.0, 'user_rejected_all_servers_out'), (90.0, 'user_rejected_all_servers_out'), (87.0, 'user_rejected_all_servers_out'), (85.0, 'user_rejected_all_servers_out'), (83.0, 'user_rejected_all_servers_out'), (82.0, 'user_rejected_all_servers_out'), (80.0, 'user_rejected_all_servers_out'), (78.0, 'user_arrival'), (78.0, 'user_rejected_all_servers_out')], 4: [(269.0, 'user_rejected_all_servers_out'), (267.0, 'user_rejected_all_servers_out'), (266.0, 'user_rejecte

In [ ]:
def handle_user_event(sum_time,current_job_events):
    
    time_states_job = None
    
    current_time = 0
    current_state = None
    for event in current_job_events:
    
        if event == "user_completed":
            print("El usuario ha completado su tarea.")
        
        elif event == "user_arrival":
            current_time = current_job_events[event]
            time_states_job = {"waiting":0,"assigned":0,"proccesing":0}
        
        elif event == "user_expired":
            pass
        
        elif event == "user_server_assigned":
            
            '''if current_state   == "assigned":
                time_states_job["assigned"] += current_job_events[event] - current_time
            elif current_state == "waiting":
                time_states_job["waiting"] += current_job_events[event] - current_time
            elif current_state == "processing":
                time_states_job["processing"] += current_job_events[event] - current_time'''
            if current_state == "None":
                current_state = "assigned"
            time_states_job[current_state] += current_job_events[event] - current_time
            current_time = current_job_events[event]
            current_state = "assigned"
        
        elif event == "user_unlock_for_resources":
            time_states_job["assigned"] += current_job_events[event] - current_time
            current_time = current_job_events[event]
            current_state = "processing"
        
        elif event == "user_rejected_not_enough_resources":
            print("Usuario rechazado: No hay suficientes recursos disponibles.")
        
        elif event == "user_rejected_not_enough_time":
            print("Usuario rechazado: No hay suficiente tiempo para completar la tarea.")
        
        elif event == "user_rejected_all_servers_out":
            print("Usuario rechazado: Todos los servidores están ocupados.")
        
        elif event == "user_interrupted":
            print("El usuario ha sido interrumpido.")
        
        else:
            print("Evento desconocido.")

# Ejemplo de uso
event = "user_server_assigned"
handle_user_event(event)
